<a href="https://colab.research.google.com/github/haskinse/bee2041_empirical_project/blob/main/source_code/03_database_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive # connect to google drive
drive.mount("/content/drive")

project_path = "/content/drive/MyDrive/bee2041_empirical_project" # define project path

Mounted at /content/drive


In [ ]:
import pandas as pd # library for data manipulation and tables

In [ ]:
artist_data = pd.read_csv(f"{project_path}/data/clean/artist_data.csv") # read clean csv data into a data frame
spotify_data = pd.read_csv(f"{project_path}/data/raw/spotify_data.csv") # read clean csv data into a data frame
rational_track_data = pd.read_csv(f"{project_path}/data/clean/track_data.csv") # read clean csv data into a data frame
# this version of tracks will be rational

In [ ]:
track_artists_rows = [] # creat list to store track to artist relationships (for many to many relationship)

for i in range(len(rational_track_data)): # for as many tracks as there are
  track_id = rational_track_data.loc[i, "track_id"] # extract track ID for the current row

  artists = spotify_data.loc[i, "artist_name"].split(",") # split comma separated artist names into a list

  for artist in artists: # for each artist
    artist = artist.strip() # remove extra whitespace from artist name

    artist_id = artist_data.loc[artist_data["artist_name"] == artist, "artist_id"].iloc[0] # look up matching artist ID from artist table

    track_artists_rows.append({"track_id": track_id, "artist_id": artist_id}) # add relationship between track and artist

rational_track_data = rational_track_data.drop(columns = "artist_name")

rational_track_data.to_csv(f"{project_path}/data/clean/rational_track_data.csv", index = False) # save new rationalised table to drive

track_artists = pd.DataFrame(track_artists_rows) # convert list of relationships into a DataFrame

track_artists.to_csv(f"{project_path}/data/clean/track_artists.csv", index = False) # save relational table to drive

track_artists.head() # view first five rows

,track_id,artist_id
0,1,23
1,2,23
2,3,23
3,4,23
4,5,23


In [ ]:
import sqlite3 # ibrary used to create and interact with SQLite databases

In [ ]:
conn = sqlite3.connect(f"{project_path}/data/clean/taylor_swift.db") # make a connection to SQLite database (creates file if it does not exist)
conn.execute("PRAGMA foreign_keys = ON;") # respect relationships between tables

In [ ]:
# the layout of the database with all of the tables included
schema = """
DROP TABLE IF EXISTS track_artists;
DROP TABLE IF EXISTS tracks;
DROP TABLE IF EXISTS albums;
DROP TABLE IF EXISTS artists;

CREATE TABLE artists (
    artist_id     INTEGER PRIMARY KEY,
    artist_name   TEXT NOT NULL UNIQUE,
    listeners     INTEGER,
    playcount     INTEGER
);

CREATE TABLE albums (
    album_id      INTEGER PRIMARY KEY,
    album_name    TEXT NOT NULL UNIQUE,
    us_peak       INTEGER,
    uk_peak       INTEGER,
    release_date  TEXT,
    label         TEXT,
    us_sales      INTEGER,
    uk_sales      INTEGER,
    riaa          TEXT,
    riaa_units    INTEGER,
    bpi           TEXT,
    bpi_units     INTEGER,
    main_genre    TEXT,
    num_genres    INTEGER,
    page_length   INTEGER,
    era           TEXT
);

CREATE TABLE tracks (
    track_id           INTEGER PRIMARY KEY,
    track_name         TEXT NOT NULL,
    album_id           INTEGER NOT NULL,
    release_date       TEXT,
    track_number       INTEGER,
    explicit           BOOLEAN,
    tempo              REAL,
    energy             REAL,
    danceability       REAL,
    duration_min       REAL,
    time_signature     TEXT,
    valence            REAL,
    acousticness       REAL,
    instrumentalness   REAL,
    liveness           REAL,
    loudness           REAL,
    speechiness        REAL,
    full_key           TEXT,
    playcount          INTEGER,
    listeners          INTEGER,
    FOREIGN KEY (album_id) REFERENCES albums(album_id)
);

CREATE TABLE track_artists (
    track_id    INTEGER,
    artist_id   INTEGER,
    PRIMARY KEY (track_id, artist_id),
    FOREIGN KEY (track_id) REFERENCES tracks(track_id),
    FOREIGN KEY (artist_id) REFERENCES artists(artist_id)
);
"""

In [ ]:
# reuploaded even previously loaded data to ensure consistency
album_data = pd.read_csv(f"{project_path}/data/clean/album_data.csv")
artist_data = pd.read_csv(f"{project_path}/data/clean/artist_data.csv")
track_artists = pd.read_csv(f"{project_path}/data/clean/track_artists.csv")
rational_track_data = pd.read_csv(f"{project_path}/data/clean/rational_track_data.csv")

In [ ]:
conn.executescript(schema) # execute SQL schema to create database tables and structure

artist_data.to_sql("artists", conn, if_exists = "append", index = False) # insert artist data into artists table

album_data.to_sql("albums", conn, if_exists = "append", index = False) # insert album data into albums table

rational_track_data.to_sql("tracks", conn, if_exists = "append", index = False) # insert track data into tracks table

# insert track to artist relationships into relational table (many-to-many relationship)
track_artists.to_sql("track_artists", conn, if_exists = "append", index = False)

conn.close() # close database connection